## Purpose:
    This script creates traveler attribute data for route prediction models. It extracts travelers whose GPS trajectories were successfully converted into
    hexagon-based trajectory files, merges traveler, companion, travel, and movement information, and converts categorical variables into one-hot encoded features.

## Input:
    - Hexagon-based GPS trajectory files in ./data/simulated_processed_inputs/Training(or Validation)/gps_hexa/
    - Traveler master data: ./data/simulated_raw_data/Jeju_data/Training(or Validation)/tn_traveller_master_여행객 Master_H.csv
    - Companion information data: ./data/simulated_raw_data/Jeju_data/Training(or Validation)/tn_companion_info_동반자정보_H.csv
    - Travel information data: ./data/simulated_raw_data/Jeju_data/Training(or Validation)/tn_travel_여행_H.csv
    - Movement history data: ./data/simulated_raw_data/Jeju_data/Training(or Validation)/tn_move_his_이동내역_H.csv
    - Optional lodge and activity history data are loaded for inspection but are not
      used in the final feature table.

## Output:
    - Traveler-level feature table saved as:
      ./data/simulated_processed_inputs/Training(or Valdiation)/properties_for_traveler.csv

## Main procedures:
    1. Load the list of hexagon-based GPS trajectory files.
    2. Extract traveler IDs and travel IDs from the file names.
    3. Load traveler, companion, travel, and movement-history datasets.
    4. Filter all attribute tables to retain only travelers with valid trajectory files.
    5. Merge traveler-level attributes into a single dataframe.
    6. Convert categorical variables into one-hot encoded features.
    7. Save the final traveler attribute table as a CSV file.

In [1]:
import numpy as np
import pandas as pd
import os
import re

## Training

### file load

In [2]:
gps_location_Tr = "./data/simulated_processed_inputs/Training/gps_hexa/"
gps_location_Va = "./data/simulated_processed_inputs/Validation/gps_hexa/"

def numerical_sort_key(file):
    return [int(t) if t.isdigit() else t for t in re.split(r'(\d+)', file)]

# Load the list of hexagon-based GPS trajectory files.
folder_Tr = './data/simulated_processed_inputs/Training/gps_hexa/'
gps_list_Tr = sorted(
    [f for f in os.listdir(folder_Tr) if f.endswith(".csv")],
    key=numerical_sort_key
)

folder_Va = './data/simulated_processed_inputs/Validation/gps_hexa/'
gps_list_Va = sorted(
    [f for f in os.listdir(folder_Va) if f.endswith(".csv")],
    key=numerical_sort_key
)

file_location_Tr = "./data/simulated_raw_data/Jeju_data/Training/"

cleaned_names = [name.replace('hexa_tn_gps_coord_', '').replace('.csv', '')[2:-2] for name in gps_list_Tr]
cleaned_names_2 = [name.replace('hexa_tn_gps_coord_', '').replace('.csv', '')[:-2] for name in gps_list_Tr]

### Data preprocessing

In [4]:
# 1. traveler Master data
traveler_col = [
    "TRAVELER_ID", "GENDER", "AGE_GRP", "MARR_STTS", "JOB_NM",
    "INCOME", "HOUSE_INCOME", "TRAVEL_NUM", "TRAVEL_TERM",
    "TRAVEL_STYL_1", "TRAVEL_STYL_2", "TRAVEL_STYL_3", "TRAVEL_STYL_4",
    "TRAVEL_STYL_5", "TRAVEL_STYL_6", "TRAVEL_STYL_7", "TRAVEL_STYL_8",
    "TRAVEL_COMPANIONS_NUM"
]
traveler = pd.read_csv(file_location_Tr + 'tn_traveller_master_여행객 Master_H.csv', encoding='utf-8')[traveler_col]
traveler = traveler[traveler["TRAVELER_ID"].isin(cleaned_names)]
# Male and Female to 0 and 1
tmp = np.zeros(len(traveler))
tmp[traveler["GENDER"] == "여"] = 1
traveler["GENDER"] = tmp
# Drop columns with null values
na_columns = traveler.columns[traveler.isnull().any()].tolist()
traveler = traveler.drop(columns= na_columns)


# 2. Companion data -> companion_df
companion_col = ["TRAVEL_ID", "REL_CD", "COMPANION_GENDER", "COMPANION_AGE_GRP"]
companion = pd.read_csv(file_location_Tr + 'tn_companion_info_동반자정보_H.csv', encoding='utf-8')[companion_col]
companion = companion[companion["TRAVEL_ID"].isin(cleaned_names_2)]
# Traveler ID format
companion["TRAVEL_ID"] = companion["TRAVEL_ID"].str[2:]
com_TRA_ID = companion["TRAVEL_ID"].unique()
companion_array = np.zeros(len(com_TRA_ID))
i = 0
for names in com_TRA_ID:
    companion_array[i] = len(companion[companion["TRAVEL_ID"] == names])
    i += 1
df_companion = pd.DataFrame({
    "TRAVELER_ID": com_TRA_ID,
    "COMPANION_COUNT": companion_array
})


# 3. travel data -> travel_df
travel_col = ["TRAVELER_ID", "TRAVEL_PURPOSE", "TRAVEL_START_YMD", "TRAVEL_END_YMD", "MVMN_NM"]
travel = pd.read_csv(file_location_Tr + 'tn_travel_여행_H.csv', encoding='utf-8')[travel_col]
travel = travel[travel["TRAVELER_ID"].isin(cleaned_names)]
# Use only the month part of the date
travel["TRAVEL_START_YMD"] = travel["TRAVEL_START_YMD"].str[5:7]
travel["TRAVEL_END_YMD"] = travel["TRAVEL_END_YMD"].str[5:7]
# travel purpose one-hot-encoding
purpose_arr = np.zeros((len(travel), 50))
for i in range(len(travel)):
    tmp_arr = list(map(int, travel.iloc[i, 1].split(';')))
    for j in tmp_arr:
        purpose_arr[i, j-1] = 1
purpose_arr_df = pd.DataFrame(purpose_arr, columns=[f'travel_purpose_{i+1}' for i in range(50)])
# dataframe
travel_df = pd.concat([travel.reset_index(drop=True), purpose_arr_df.reset_index(drop=True)], axis=1)
travel_df = travel_df.drop("TRAVEL_PURPOSE", axis=1)


# 4. movement data -> No need to use all the movement data, just the most used transportation mode and the start code.
move_col = ["TRAVEL_ID", "MVMN_CD_1", "START_VISIT_AREA_ID", "END_VISIT_AREA_ID"]
move = pd.read_csv(file_location_Tr + 'tn_move_his_이동내역_H.csv', encoding='utf-8')[move_col]
# traveler ID format
move["TRAVEL_ID"] = move["TRAVEL_ID"].str[2:]
move = move[move["TRAVEL_ID"].isin(cleaned_names)]
# For each travel ID, fill the start code with the first start code of that travel ID
move_ID = move["TRAVEL_ID"].unique()
for i in range(len(move_ID)):
    move.loc[move["TRAVEL_ID"] == move_ID[i], move.columns[2]] = move[move["TRAVEL_ID"] == move_ID[i]].iloc[0, 2]
move = move = move.dropna(subset=["END_VISIT_AREA_ID"])
# For each travel ID, fill the most used transportation mode with the most used transportation mode of that travel ID
mode_mvmn = move.groupby('TRAVEL_ID')['MVMN_CD_1'].agg(lambda x: x.value_counts().idxmax())
mode_mvmn_df = mode_mvmn.reset_index().rename(columns={'MVMN_CD_1': 'MOST_MVMN_CD_1'})
mode_mvmn_df.rename(columns={"TRAVEL_ID": "TRAVELER_ID"}, inplace=True)


# 5. Lodge consumption data and can't use
lodge_col = ["TRAVEL_ID", "ROAD_NM_ADDR", "LOTNO_ADDR", "ROAD_NM_CD", "LOTNO_CD"]
lodge = pd.read_csv(file_location_Tr + 'tn_lodge_consume_his_숙박소비내역_H.csv', encoding='utf-8')[lodge_col]
lodge["TRAVEL_ID"] = lodge["TRAVEL_ID"].str[2:]
lodge = lodge[lodge["TRAVEL_ID"].isin(cleaned_names)]


# 6. Activity data and can't use
activity_col = ["TRAVEL_ID", "VISIT_AREA_ID", "ACTIVITY_TYPE_SEQ"]
activity = pd.read_csv(file_location_Tr + 'tn_activity_his_활동내역_H.csv', encoding='utf-8')[activity_col]
# Traveler ID format
activity["TRAVEL_ID"] = activity["TRAVEL_ID"].str[2:]
activity = activity[activity["TRAVEL_ID"].isin(cleaned_names)]

### make data frame

In [5]:
df_tmp = pd.merge(traveler, df_companion, on='TRAVELER_ID', how='left')
df_tmp_2 = pd.merge(df_tmp, travel_df, on='TRAVELER_ID', how='left')
df_final = pd.merge(df_tmp_2, mode_mvmn_df, on='TRAVELER_ID', how='left')
df_final["COMPANION_COUNT"] = df_final["COMPANION_COUNT"].fillna(0)
df_final = df_final.drop(columns=["MVMN_NM", "COMPANION_COUNT"])


# 'MARR_STTS' one-hot-encoding
num_class = 5
tmp = np.zeros((len(df_final), num_class))
for i in range(len(df_final)):
    value = df_final['MARR_STTS'][i]
    if not pd.isna(value):
        tmp[i, int(value)-1] = 1 # index is value-1 because the value is 1~5 but index starts from 0
one_hot_df = pd.DataFrame(tmp, columns=[f"MARR_STTS_{i}" for i in range(1, num_class+1)])
df_final = pd.concat([df_final.reset_index(drop=True), one_hot_df.reset_index(drop=True)], axis=1)
df_final = df_final.drop(columns='MARR_STTS')

# 'JOB_NM' one-hot
num_class = 13
tmp = np.zeros((len(df_final), num_class))
for i in range(len(df_final)):
    value = df_final['JOB_NM'][i]
    if not pd.isna(value):
        tmp[i, int(value)-1] = 1
one_hot_df = pd.DataFrame(tmp, columns=[f"JOB_NM_{i}" for i in range(1, num_class+1)])
df_final = pd.concat([df_final.reset_index(drop=True), one_hot_df.reset_index(drop=True)], axis=1)
df_final = df_final.drop(columns='JOB_NM')

# 'TRAVEL_STYL_1' ~ 'TRAVEL_STYL_8' one-hot
style_cols = [f'TRAVEL_STYL_{i}' for i in range(1, 9)]
num_class = 7
for col in style_cols:
    tmp = np.zeros((len(df_final), num_class))
    for i in range(len(df_final)):
        value = df_final[col][i]
        if not pd.isna(value):
            tmp[i, int(value)-1] = 1
    one_hot_df = pd.DataFrame(tmp, columns=[f"{col}_{i}" for i in range(1, num_class+1)])
    df_final = pd.concat([df_final.reset_index(drop=True), one_hot_df.reset_index(drop=True)], axis=1)
    df_final = df_final.drop(columns=col)

# 'TRAVEL_START_YMD' one-hot
num_class = 12
tmp = np.zeros((len(df_final), num_class))
for i in range(len(df_final)):
    value = df_final['TRAVEL_START_YMD'][i]
    if not pd.isna(value):
        tmp[i, int(value)-1] = 1
one_hot_df = pd.DataFrame(tmp, columns=[f"TRAVEL_START_YMD_{i}" for i in range(1, num_class+1)])
df_final = pd.concat([df_final.reset_index(drop=True), one_hot_df.reset_index(drop=True)], axis=1)
df_final = df_final.drop(columns='TRAVEL_START_YMD')

# 'TRAVEL_END_YMD' one-hot
tmp = np.zeros((len(df_final), num_class))
for i in range(len(df_final)):
    value = df_final['TRAVEL_END_YMD'][i]
    if not pd.isna(value):
        tmp[i, int(value)-1] = 1
one_hot_df = pd.DataFrame(tmp, columns=[f"TRAVEL_END_YMD_{i}" for i in range(1, num_class+1)])
df_final = pd.concat([df_final.reset_index(drop=True), one_hot_df.reset_index(drop=True)], axis=1)
df_final = df_final.drop(columns='TRAVEL_END_YMD')

# 'MOST_MVMN_CD_1' one-hot
num_class = 16
tmp = np.zeros((len(df_final), num_class))
for i in range(len(df_final)):
    value = df_final['MOST_MVMN_CD_1'][i]
    if not pd.isna(value):
        tmp[i, int(value)-1] = 1
one_hot_df = pd.DataFrame(tmp, columns=[f"MOST_MVMN_CD_1_{i}" for i in range(1, num_class+1)])
df_final = pd.concat([df_final.reset_index(drop=True), one_hot_df.reset_index(drop=True)], axis=1)
df_final = df_final.drop(columns='MOST_MVMN_CD_1')


df_final.to_csv("./data/simulated_processed_inputs/Training/properties_for_traveler.csv")

In [7]:
df_final

,TRAVELER_ID,GENDER,AGE_GRP,INCOME,TRAVEL_NUM,TRAVEL_TERM,TRAVEL_COMPANIONS_NUM,travel_purpose_1,travel_purpose_2,travel_purpose_3,...,MOST_MVMN_CD_1_7,MOST_MVMN_CD_1_8,MOST_MVMN_CD_1_9,MOST_MVMN_CD_1_10,MOST_MVMN_CD_1_11,MOST_MVMN_CD_1_12,MOST_MVMN_CD_1_13,MOST_MVMN_CD_1_14,MOST_MVMN_CD_1_15,MOST_MVMN_CD_1_16
0,g003414,1.0,50,4,4,2,1,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,h002865,1.0,20,4,1,3,5,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,h000468,1.0,30,6,1,2,5,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,h006653,0.0,20,6,6,2,1,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,h000032,1.0,50,1,5,3,1,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147,g000216,0.0,30,4,2,3,1,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
148,h005884,0.0,50,2,2,2,0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
149,h001629,1.0,40,4,1,3,0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
150,e004260,0.0,20,4,10,3,3,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


## Validation
### same code with training

### file load

In [8]:
file_location_Va = "./data/simulated_raw_data/Jeju_data/Validation/"

cleaned_names = [name.replace('hexa_tn_gps_coord_', '').replace('.csv', '')[2:-2] for name in gps_list_Va]
cleaned_names_2 = [name.replace('hexa_tn_gps_coord_', '').replace('.csv', '')[:-2] for name in gps_list_Va]

### Data preprocessing

In [10]:
traveler_col = [
    "TRAVELER_ID", "GENDER", "AGE_GRP", "MARR_STTS", "JOB_NM",
    "INCOME", "HOUSE_INCOME", "TRAVEL_NUM", "TRAVEL_TERM",
    "TRAVEL_STYL_1", "TRAVEL_STYL_2", "TRAVEL_STYL_3", "TRAVEL_STYL_4",
    "TRAVEL_STYL_5", "TRAVEL_STYL_6", "TRAVEL_STYL_7", "TRAVEL_STYL_8",
    "TRAVEL_COMPANIONS_NUM"
]
traveler = pd.read_csv(file_location_Va + 'tn_traveller_master_여행객 Master_H.csv', encoding='utf-8')[traveler_col]
traveler = traveler[traveler["TRAVELER_ID"].isin(cleaned_names)]
tmp = np.zeros(len(traveler))
tmp[traveler["GENDER"] == "여"] = 1 # Gender: Female
traveler["GENDER"] = tmp
na_columns = traveler.columns[traveler.isnull().any()].tolist()
traveler = traveler.drop(columns= na_columns)


companion_col = ["TRAVEL_ID", "REL_CD", "COMPANION_GENDER", "COMPANION_AGE_GRP"]
companion = pd.read_csv(file_location_Va + 'tn_companion_info_동반자정보_H.csv', encoding='utf-8')[companion_col]
companion = companion[companion["TRAVEL_ID"].isin(cleaned_names_2)]
companion["TRAVEL_ID"] = companion["TRAVEL_ID"].str[2:]
com_TRA_ID = companion["TRAVEL_ID"].unique()
companion_array = np.zeros(len(com_TRA_ID))
i = 0
for names in com_TRA_ID:
    companion_array[i] = len(companion[companion["TRAVEL_ID"] == names])
    i += 1
df_companion = pd.DataFrame({
    "TRAVELER_ID": com_TRA_ID,
    "COMPANION_COUNT": companion_array
})


travel_col = ["TRAVELER_ID", "TRAVEL_PURPOSE", "TRAVEL_START_YMD", "TRAVEL_END_YMD", "MVMN_NM"]
travel = pd.read_csv(file_location_Va + 'tn_travel_여행_H.csv', encoding='utf-8')[travel_col]
travel = travel[travel["TRAVELER_ID"].isin(cleaned_names)]
travel["TRAVEL_START_YMD"] = travel["TRAVEL_START_YMD"].str[5:7]
travel["TRAVEL_END_YMD"] = travel["TRAVEL_END_YMD"].str[5:7]
purpose_arr = np.zeros((len(travel), 50))
for i in range(len(travel)):
    tmp_arr = list(map(int, travel.iloc[i, 1].split(';')))
    for j in tmp_arr:
        purpose_arr[i, j-1] = 1
purpose_arr_df = pd.DataFrame(purpose_arr, columns=[f'travel_purpose_{i+1}' for i in range(50)])
travel_df = pd.concat([travel.reset_index(drop=True), purpose_arr_df.reset_index(drop=True)], axis=1)
travel_df = travel_df.drop("TRAVEL_PURPOSE", axis=1)


move_col = ["TRAVEL_ID", "MVMN_CD_1", "START_VISIT_AREA_ID", "END_VISIT_AREA_ID"]
move = pd.read_csv(file_location_Va + 'tn_move_his_이동내역_H.csv', encoding='utf-8')[move_col]
move["TRAVEL_ID"] = move["TRAVEL_ID"].str[2:]
move = move[move["TRAVEL_ID"].isin(cleaned_names)]
move_ID = move["TRAVEL_ID"].unique()
for i in range(len(move_ID)):
    move.loc[move["TRAVEL_ID"] == move_ID[i], move.columns[2]] = move[move["TRAVEL_ID"] == move_ID[i]].iloc[0, 2]
move = move = move.dropna(subset=["END_VISIT_AREA_ID"])
mode_mvmn = move.groupby('TRAVEL_ID')['MVMN_CD_1'].agg(lambda x: x.value_counts().idxmax())
mode_mvmn_df = mode_mvmn.reset_index().rename(columns={'MVMN_CD_1': 'MOST_MVMN_CD_1'})
mode_mvmn_df.rename(columns={"TRAVEL_ID": "TRAVELER_ID"}, inplace=True)


lodge_col = ["TRAVEL_ID", "ROAD_NM_ADDR", "LOTNO_ADDR", "ROAD_NM_CD", "LOTNO_CD"]
lodge = pd.read_csv(file_location_Va + 'tn_lodge_consume_his_숙박소비내역_H.csv', encoding='utf-8')[lodge_col]
lodge["TRAVEL_ID"] = lodge["TRAVEL_ID"].str[2:]
lodge = lodge[lodge["TRAVEL_ID"].isin(cleaned_names)]


activity_col = ["TRAVEL_ID", "VISIT_AREA_ID", "ACTIVITY_TYPE_SEQ"]
activity = pd.read_csv(file_location_Va + 'tn_activity_his_활동내역_H.csv', encoding='utf-8')[activity_col]
# Traveler ID format
activity["TRAVEL_ID"] = activity["TRAVEL_ID"].str[2:]
activity = activity[activity["TRAVEL_ID"].isin(cleaned_names)]

### make dataframe

In [11]:
df_tmp = pd.merge(traveler, df_companion, on='TRAVELER_ID', how='left')
df_tmp_2 = pd.merge(df_tmp, travel_df, on='TRAVELER_ID', how='left')
df_final = pd.merge(df_tmp_2, mode_mvmn_df, on='TRAVELER_ID', how='left')
df_final["COMPANION_COUNT"] = df_final["COMPANION_COUNT"].fillna(0)
df_final = df_final.drop(columns=["MVMN_NM", "COMPANION_COUNT"])


num_class = 5
tmp = np.zeros((len(df_final), num_class))
for i in range(len(df_final)):
    value = df_final['MARR_STTS'][i]
    if not pd.isna(value):  
        tmp[i, int(value)-1] = 1 
one_hot_df = pd.DataFrame(tmp, columns=[f"MARR_STTS_{i}" for i in range(1, num_class+1)])
df_final = pd.concat([df_final.reset_index(drop=True), one_hot_df.reset_index(drop=True)], axis=1)
df_final = df_final.drop(columns='MARR_STTS')


num_class = 13
tmp = np.zeros((len(df_final), num_class))
for i in range(len(df_final)):
    value = df_final['JOB_NM'][i]
    if not pd.isna(value):
        tmp[i, int(value)-1] = 1
one_hot_df = pd.DataFrame(tmp, columns=[f"JOB_NM_{i}" for i in range(1, num_class+1)])
df_final = pd.concat([df_final.reset_index(drop=True), one_hot_df.reset_index(drop=True)], axis=1)
df_final = df_final.drop(columns='JOB_NM')


style_cols = [f'TRAVEL_STYL_{i}' for i in range(1, 9)]
num_class = 7
for col in style_cols:
    tmp = np.zeros((len(df_final), num_class))
    for i in range(len(df_final)):
        value = df_final[col][i]
        if not pd.isna(value):
            tmp[i, int(value)-1] = 1
    one_hot_df = pd.DataFrame(tmp, columns=[f"{col}_{i}" for i in range(1, num_class+1)])
    df_final = pd.concat([df_final.reset_index(drop=True), one_hot_df.reset_index(drop=True)], axis=1)
    df_final = df_final.drop(columns=col)


num_class = 12
tmp = np.zeros((len(df_final), num_class))
for i in range(len(df_final)):
    value = df_final['TRAVEL_START_YMD'][i]
    if not pd.isna(value):
        tmp[i, int(value)-1] = 1
one_hot_df = pd.DataFrame(tmp, columns=[f"TRAVEL_START_YMD_{i}" for i in range(1, num_class+1)])
df_final = pd.concat([df_final.reset_index(drop=True), one_hot_df.reset_index(drop=True)], axis=1)
df_final = df_final.drop(columns='TRAVEL_START_YMD')


tmp = np.zeros((len(df_final), num_class))
for i in range(len(df_final)):
    value = df_final['TRAVEL_END_YMD'][i]
    if not pd.isna(value):
        tmp[i, int(value)-1] = 1
one_hot_df = pd.DataFrame(tmp, columns=[f"TRAVEL_END_YMD_{i}" for i in range(1, num_class+1)])
df_final = pd.concat([df_final.reset_index(drop=True), one_hot_df.reset_index(drop=True)], axis=1)
df_final = df_final.drop(columns='TRAVEL_END_YMD')


num_class = 16
tmp = np.zeros((len(df_final), num_class))
for i in range(len(df_final)):
    value = df_final['MOST_MVMN_CD_1'][i]
    if not pd.isna(value):
        tmp[i, int(value)-1] = 1
one_hot_df = pd.DataFrame(tmp, columns=[f"MOST_MVMN_CD_1_{i}" for i in range(1, num_class+1)])
df_final = pd.concat([df_final.reset_index(drop=True), one_hot_df.reset_index(drop=True)], axis=1)
df_final = df_final.drop(columns='MOST_MVMN_CD_1')


df_final.to_csv("./data/simulated_processed_inputs/Validation/properties_for_traveler.csv")